# Frame Co-occurrence Analysis

Analyzing which frames appear together in articles, globally and by media outlet.

In [ ]:
!pip install pandas numpy matplotlib seaborn pyarrow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

In [ ]:
df = pd.read_parquet('data/merged_topic_and_frames_filtered.parquet')
print(f'Loaded {len(df):,} articles')

frames = ['economic', 'fairness', 'public_op', 'political', 'quality_life', 
          'crime', 'culture', 'health', 'legality', 'morality', 
          'policy', 'regulation', 'security', 'cap&res']

frame_matrix = np.array(df['vector'].tolist())

## Global Frame Co-occurrence

In [ ]:
cooccur = frame_matrix.T @ frame_matrix

pairs = []
for i, j in combinations(range(len(frames)), 2):
    pairs.append({
        'frame_1': frames[i],
        'frame_2': frames[j],
        'count': cooccur[i, j],
        'rate': cooccur[i, j] / len(df)
    })

pairs_df = pd.DataFrame(pairs).sort_values('count', ascending=False)
print('Top 10 co-occurring frame pairs:')
pairs_df.head(10)

## Frame Co-occurrence by Media Outlet

In [ ]:
print('Articles per outlet:')
print(df['outlet_name'].value_counts())

In [ ]:
top_pairs = pairs_df.head(5)

comparison = []
for outlet in df['outlet_name'].unique():
    subset = df[df['outlet_name'] == outlet]
    mat = np.array(subset['vector'].tolist())
    outlet_cooccur = mat.T @ mat
    
    row = {'outlet': outlet, 'n_articles': len(subset)}
    for _, pair in top_pairs.iterrows():
        i, j = frames.index(pair['frame_1']), frames.index(pair['frame_2'])
        pair_name = f"{pair['frame_1']}-{pair['frame_2']}"
        row[pair_name] = outlet_cooccur[i, j] / len(subset)
    comparison.append(row)

comparison_df = pd.DataFrame(comparison).sort_values('n_articles', ascending=False)
comparison_df

In [ ]:
pair_cols = [c for c in comparison_df.columns if '-' in c]

fig, ax = plt.subplots(figsize=(12, 6))
comparison_df.set_index('outlet')[pair_cols].plot(kind='bar', ax=ax)
plt.title('Top Frame Pair Co-occurrence Rates by Outlet')
plt.ylabel('Co-occurrence Rate')
plt.legend(title='Frame Pair', bbox_to_anchor=(1.02, 1))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
print('Co-occurrence rate ranges across outlets:\n')
for col in pair_cols:
    vals = comparison_df[col]
    print(f'{col}: {vals.min():.0%} - {vals.max():.0%} (range: {vals.max()-vals.min():.0%})')